# S5-03: 건축공학 MCP 서버 — BIM + 구조 계산 + 통합
**구조공학 도메인 종합 실습**

## 학습 목표
- BIM(IFC) 모델 데이터를 MCP 리소스로 제공하는 서버를 구현한다
- 구조 설계 검토 계산을 MCP 도구로 구현한다
- Tools + Resources + Prompts를 통합한 건축공학 MCP 서버를 완성한다
- MCP 서버 파일을 작성하고 Inspector로 테스트한다

## 사전 준비
1. S5-01, S5-02 완료
2. FastMCP 설치 확인: `pip install "mcp[cli]"`

In [ ]:
# 패키지 확인
%pip install "mcp[cli]" pydantic

In [ ]:
import math
import json
from mcp.server.fastmcp import FastMCP
from pydantic import BaseModel, Field

## 배경: 건축공학에서 MCP의 가치

건축 프로젝트에서 AI가 접근해야 하는 데이터:

| 데이터 소스 | MCP Primitive | 예시 |
| --- | --- | --- |
| BIM 모델 (IFC) | **Resources** | 층 구성, 부재 목록, 물량 |
| 구조 계산 엔진 | **Tools** | 모멘트 계산, 전단 검토, 철근량 산정 |
| 설계 기준 (KDS) | **Resources + Prompts** | 조문 내용 + 검토 프롬프트 |

이 실습에서는 이 세 가지를 **하나의 MCP 서버**로 통합한다.

---
## 연습 1: BIM 데이터 MCP 서버

### 문제
건물 BIM 모델의 데이터를 MCP 리소스로 제공하는 서버를 구현하세요.

**건물 데이터 (시연용):**
```python
BUILDING = {
    "project": "경희대 연구동",
    "stories": [
        {"name": "1F", "level": 0.0, "height": 4.5},
        {"name": "2F", "level": 4.5, "height": 3.6},
        {"name": "3F", "level": 8.1, "height": 3.6},
    ],
    "beams": [
        {"id": "B1", "floor": "2F", "span": 8.0, "b": 350, "h": 600, "fck": 27},
        {"id": "B2", "floor": "2F", "span": 6.0, "b": 300, "h": 500, "fck": 27},
        {"id": "B3", "floor": "3F", "span": 8.0, "b": 350, "h": 600, "fck": 24},
    ],
    "columns": [
        {"id": "C1", "floor": "1F", "b": 500, "h": 500, "fck": 30, "Pu": 3000},
        {"id": "C2", "floor": "1F", "b": 400, "h": 400, "fck": 27, "Pu": 1800},
    ]
}
```

**요구사항:**
1. `bim://overview` — 건물 개요 (프로젝트명, 층수, 보/기둥 개수)
2. `bim://stories` — 층 구성 정보
3. `bim://beams/{floor}` — 특정 층의 보 목록
4. `bim://columns/{floor}` — 특정 층의 기둥 목록
5. 도구: `calc_concrete_volume(member_type: str, floor: str)` — 콘크리트 물량 산출

**기대 출력:**
```
bim://overview → "경희대 연구동: 3개 층, 보 3개, 기둥 2개"
bim://beams/2F → "2F층 보: B1(8m, 350x600), B2(6m, 300x500)"
calc_concrete_volume("beam", "2F") → {"volume_m3": ..., "count": 2}
```

In [ ]:
# TODO: 아래 코드를 완성하세요

bim_mcp = FastMCP("bim-data-server")

BUILDING = {
    "project": "경희대 연구동",
    "stories": [
        {"name": "1F", "level": 0.0, "height": 4.5},
        {"name": "2F", "level": 4.5, "height": 3.6},
        {"name": "3F", "level": 8.1, "height": 3.6},
    ],
    "beams": [
        {"id": "B1", "floor": "2F", "span": 8.0, "b": 350, "h": 600, "fck": 27},
        {"id": "B2", "floor": "2F", "span": 6.0, "b": 300, "h": 500, "fck": 27},
        {"id": "B3", "floor": "3F", "span": 8.0, "b": 350, "h": 600, "fck": 24},
    ],
    "columns": [
        {"id": "C1", "floor": "1F", "b": 500, "h": 500, "fck": 30, "Pu": 3000},
        {"id": "C2", "floor": "1F", "b": 400, "h": 400, "fck": 27, "Pu": 1800},
    ]
}

# TODO: 리소스 4개 + 도구 1개 정의

In [ ]:
# === 솔루션 ===

bim_mcp = FastMCP("bim-data-server")

BUILDING = {
    "project": "경희대 연구동",
    "stories": [
        {"name": "1F", "level": 0.0, "height": 4.5},
        {"name": "2F", "level": 4.5, "height": 3.6},
        {"name": "3F", "level": 8.1, "height": 3.6},
    ],
    "beams": [
        {"id": "B1", "floor": "2F", "span": 8.0, "b": 350, "h": 600, "fck": 27},
        {"id": "B2", "floor": "2F", "span": 6.0, "b": 300, "h": 500, "fck": 27},
        {"id": "B3", "floor": "3F", "span": 8.0, "b": 350, "h": 600, "fck": 24},
    ],
    "columns": [
        {"id": "C1", "floor": "1F", "b": 500, "h": 500, "fck": 30, "Pu": 3000},
        {"id": "C2", "floor": "1F", "b": 400, "h": 400, "fck": 27, "Pu": 1800},
    ]
}

@bim_mcp.resource("bim://overview")
def get_building_overview() -> str:
    """건물 개요 정보를 반환합니다."""
    return (
        f"{BUILDING['project']}: "
        f"{len(BUILDING['stories'])}개 층, "
        f"보 {len(BUILDING['beams'])}개, "
        f"기둥 {len(BUILDING['columns'])}개"
    )

@bim_mcp.resource("bim://stories")
def get_stories() -> str:
    """층 구성 정보를 반환합니다."""
    lines = ["층 구성:"]
    for s in BUILDING["stories"]:
        lines.append(f"  {s['name']}: GL{s['level']:+.1f}m, 층고 {s['height']}m")
    return "\n".join(lines)

@bim_mcp.resource("bim://beams/{floor}")
def get_beams_by_floor(floor: str) -> str:
    """특정 층의 보 목록을 반환합니다.

    Args:
        floor: 층 이름 (예: 2F)
    """
    beams = [b for b in BUILDING["beams"] if b["floor"] == floor.upper()]
    if not beams:
        return f"{floor}층에 등록된 보가 없습니다."
    lines = [f"{floor}층 보 목록 ({len(beams)}개):"]
    for b in beams:
        lines.append(
            f"  {b['id']}: 경간 {b['span']}m, "
            f"단면 {b['b']}x{b['h']}mm, fck={b['fck']}MPa"
        )
    return "\n".join(lines)

@bim_mcp.resource("bim://columns/{floor}")
def get_columns_by_floor(floor: str) -> str:
    """특정 층의 기둥 목록을 반환합니다.

    Args:
        floor: 층 이름 (예: 1F)
    """
    cols = [c for c in BUILDING["columns"] if c["floor"] == floor.upper()]
    if not cols:
        return f"{floor}층에 등록된 기둥이 없습니다."
    lines = [f"{floor}층 기둥 목록 ({len(cols)}개):"]
    for c in cols:
        lines.append(
            f"  {c['id']}: 단면 {c['b']}x{c['h']}mm, "
            f"fck={c['fck']}MPa, Pu={c['Pu']}kN"
        )
    return "\n".join(lines)

@bim_mcp.tool()
def calc_concrete_volume(member_type: str, floor: str = "all") -> dict:
    """부재별 콘크리트 물량을 산출합니다.

    Args:
        member_type: 부재 유형 (beam 또는 column)
        floor: 층 이름 (기본: all)
    """
    if member_type.lower() == "beam":
        members = BUILDING["beams"]
        if floor.lower() != "all":
            members = [m for m in members if m["floor"] == floor.upper()]
        # 보 체적: b * h * span (단위 변환: mm→m)
        total = sum(m["b"]/1000 * m["h"]/1000 * m["span"] for m in members)
    elif member_type.lower() == "column":
        members = BUILDING["columns"]
        if floor.lower() != "all":
            members = [m for m in members if m["floor"] == floor.upper()]
        # 기둥 체적: b * h * 층고
        story_heights = {s["name"]: s["height"] for s in BUILDING["stories"]}
        total = sum(
            m["b"]/1000 * m["h"]/1000 * story_heights.get(m["floor"], 3.6)
            for m in members
        )
    else:
        return {"error": "지원: beam, column"}

    return {
        "member_type": member_type,
        "floor": floor,
        "count": len(members),
        "volume_m3": round(total, 3)
    }

# 테스트
print("=== 건물 개요 ===")
print(get_building_overview())
print("\n=== 층 구성 ===")
print(get_stories())
print("\n=== 2F 보 목록 ===")
print(get_beams_by_floor("2F"))
print("\n=== 1F 기둥 목록 ===")
print(get_columns_by_floor("1F"))
print("\n=== 보 콘크리트 물량 (2F) ===")
print(calc_concrete_volume("beam", "2F"))
print("\n=== 기둥 콘크리트 물량 (전체) ===")
print(calc_concrete_volume("column", "all"))

In [ ]:
# 검증 함수
def verify_exercise1():
    passed = 0

    # 개요 리소스
    overview = get_building_overview()
    if "3개 층" in overview and "보 3개" in overview:
        print("  PASS: 건물 개요 정확")
        passed += 1
    else:
        print(f"  FAIL: 건물 개요 — {overview}")

    # 층 정보
    stories = get_stories()
    if "1F" in stories and "2F" in stories and "3F" in stories:
        print("  PASS: 층 정보 포함")
        passed += 1
    else:
        print("  FAIL: 층 정보 누락")

    # 보 목록
    beams_2f = get_beams_by_floor("2F")
    if "B1" in beams_2f and "B2" in beams_2f and "B3" not in beams_2f:
        print("  PASS: 2F 보 필터링 정확")
        passed += 1
    else:
        print(f"  FAIL: 2F 보 필터링 — {beams_2f}")

    # 기둥 목록
    cols_1f = get_columns_by_floor("1F")
    if "C1" in cols_1f and "C2" in cols_1f:
        print("  PASS: 1F 기둥 목록 정확")
        passed += 1
    else:
        print("  FAIL: 1F 기둥 목록 오류")

    # 물량 산출
    vol = calc_concrete_volume("beam", "2F")
    # B1: 0.35*0.6*8 = 1.68, B2: 0.3*0.5*6 = 0.9, total = 2.58
    if abs(vol["volume_m3"] - 2.58) < 0.01:
        print(f"  PASS: 보 물량 = {vol['volume_m3']} m3")
        passed += 1
    else:
        print(f"  FAIL: 보 물량 — expected 2.58, got {vol['volume_m3']}")

    # 기둥 물량
    vol_col = calc_concrete_volume("column", "1F")
    # C1: 0.5*0.5*4.5=1.125, C2: 0.4*0.4*4.5=0.72, total=1.845
    if abs(vol_col["volume_m3"] - 1.845) < 0.01:
        print(f"  PASS: 기둥 물량 = {vol_col['volume_m3']} m3")
        passed += 1
    else:
        print(f"  FAIL: 기둥 물량 — expected 1.845, got {vol_col['volume_m3']}")

    print(f"\n결과: {passed}/6 통과")
    return passed == 6

verify_exercise1()

---
## 연습 2: 구조 설계 검토 MCP 서버

### 문제
RC 보와 기둥의 설계 적정성을 검토하는 MCP 도구 서버를 구현하세요.

**요구 도구:**
1. `check_beam_flexure(b, d, fck, fy, As, Mu)` — 보 휨 검토
   - a = As*fy / (0.85*fck*b)
   - phi_Mn = 0.85 * As*fy*(d-a/2) / 1e6
   - 판정: phi_Mn >= Mu이면 OK

2. `check_beam_shear(b, d, fck, fy, Vu, stirrup_d, stirrup_s)` — 보 전단 검토
   - Vc = (1/6)*sqrt(fck)*b*d / 1000
   - Av = 2 * pi * (stirrup_d/2)^2  (2개 다리)
   - Vs = Av*fy*d / (stirrup_s*1000)
   - phi_Vn = 0.75*(Vc+Vs)
   - 판정: phi_Vn >= Vu이면 OK

3. `check_column_axial(b, h, fck, fy, Ast, Pu)` — 기둥 축력 검토
   - Ag = b*h
   - rho = Ast/Ag (0.01~0.08 범위 확인)
   - phi_Pn_max = 0.80 * 0.65 * (0.85*fck*(Ag-Ast) + fy*Ast) / 1000
   - 판정: Pu <= phi_Pn_max이면 OK

**기대 출력:**
```python
check_beam_flexure(350, 560, 27, 400, 2534, 380)
# → {"phi_Mn_kNm": ~423, "Mu_kNm": 380, "check": "OK", ...}
```

In [ ]:
# TODO: 아래 코드를 완성하세요

design_mcp = FastMCP("structural-design-check")

@design_mcp.tool()
def check_beam_flexure(
    b: float, d: float, fck: float, fy: float, As: float, Mu: float
) -> dict:
    """RC 보의 휨 강도를 검토합니다 (KDS 14 20 20)."""
    # TODO: 계산 구현
    pass

@design_mcp.tool()
def check_beam_shear(
    b: float, d: float, fck: float, fy: float,
    Vu: float, stirrup_d: float, stirrup_s: float
) -> dict:
    """RC 보의 전단 강도를 검토합니다 (KDS 14 20 22)."""
    # TODO: 계산 구현
    pass

@design_mcp.tool()
def check_column_axial(
    b: float, h: float, fck: float, fy: float, Ast: float, Pu: float
) -> dict:
    """RC 기둥의 축력 강도를 검토합니다 (KDS 14 20 20)."""
    # TODO: 계산 구현
    pass

In [ ]:
# === 솔루션 ===

design_mcp = FastMCP("structural-design-check")

@design_mcp.tool()
def check_beam_flexure(
    b: float, d: float, fck: float, fy: float, As: float, Mu: float
) -> dict:
    """RC 보의 휨 강도를 검토합니다 (KDS 14 20 20).

    Args:
        b: 단면 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 설계기준강도 (MPa)
        fy: 철근 항복강도 (MPa)
        As: 인장 철근량 (mm2)
        Mu: 설계 휨모멘트 (kN-m)
    """
    a = (As * fy) / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn

    return {
        "a_mm": round(a, 1),
        "Mn_kNm": round(Mn, 1),
        "phi_Mn_kNm": round(phi_Mn, 1),
        "Mu_kNm": Mu,
        "check": "OK" if phi_Mn >= Mu else "NG",
        "utilization": round(Mu / phi_Mn, 3) if phi_Mn > 0 else None,
        "reference": "KDS 14 20 20, 4.2.3"
    }

@design_mcp.tool()
def check_beam_shear(
    b: float, d: float, fck: float, fy: float,
    Vu: float, stirrup_d: float, stirrup_s: float
) -> dict:
    """RC 보의 전단 강도를 검토합니다 (KDS 14 20 22).

    Args:
        b: 단면 폭 (mm)
        d: 유효 깊이 (mm)
        fck: 콘크리트 설계기준강도 (MPa)
        fy: 철근 항복강도 (MPa)
        Vu: 설계 전단력 (kN)
        stirrup_d: 전단철근 직경 (mm, 예: 10 for D10)
        stirrup_s: 전단철근 간격 (mm)
    """
    Vc = (1/6) * math.sqrt(fck) * b * d / 1000

    Av = 2 * math.pi * (stirrup_d / 2) ** 2  # 2개 다리
    Vs = Av * fy * d / (stirrup_s * 1000)

    phi_Vn = 0.75 * (Vc + Vs)

    return {
        "Vc_kN": round(Vc, 1),
        "Av_mm2": round(Av, 1),
        "Vs_kN": round(Vs, 1),
        "phi_Vn_kN": round(phi_Vn, 1),
        "Vu_kN": Vu,
        "check": "OK" if phi_Vn >= Vu else "NG",
        "reference": "KDS 14 20 22, 5.3-5.4"
    }

@design_mcp.tool()
def check_column_axial(
    b: float, h: float, fck: float, fy: float, Ast: float, Pu: float
) -> dict:
    """RC 기둥의 축력 강도를 검토합니다 (KDS 14 20 20).

    Args:
        b: 기둥 폭 (mm)
        h: 기둥 깊이 (mm)
        fck: 콘크리트 설계기준강도 (MPa)
        fy: 철근 항복강도 (MPa)
        Ast: 총 주근 단면적 (mm2)
        Pu: 설계 축력 (kN)
    """
    Ag = b * h
    rho = Ast / Ag
    phi_Pn_max = 0.80 * 0.65 * (0.85 * fck * (Ag - Ast) + fy * Ast) / 1000
    axial_ratio = Pu / (fck * Ag / 1000)

    return {
        "Ag_mm2": Ag,
        "rho": round(rho, 4),
        "rho_check": "OK" if 0.01 <= rho <= 0.08 else "NG",
        "phi_Pn_max_kN": round(phi_Pn_max, 1),
        "Pu_kN": Pu,
        "axial_check": "OK" if Pu <= phi_Pn_max else "NG",
        "axial_ratio": round(axial_ratio, 3),
        "reference": "KDS 14 20 20"
    }

# 테스트
print("=== 보 휨 검토 ===")
r1 = check_beam_flexure(350, 560, 27, 400, 2534, 380)
print(json.dumps(r1, indent=2, ensure_ascii=False))

print("\n=== 보 전단 검토 (D10@200) ===")
r2 = check_beam_shear(350, 560, 27, 400, 250, 10, 200)
print(json.dumps(r2, indent=2, ensure_ascii=False))

print("\n=== 기둥 축력 검토 ===")
# 8-D25 = 8 * 506.7 = 4053.6 mm2
r3 = check_column_axial(500, 500, 30, 400, 4053.6, 3000)
print(json.dumps(r3, indent=2, ensure_ascii=False))

In [ ]:
# 검증 함수
def verify_exercise2():
    passed = 0

    # 보 휨 검토
    r1 = check_beam_flexure(350, 560, 27, 400, 2534, 380)
    if r1["check"] == "OK" and r1["phi_Mn_kNm"] > 380:
        print(f"  PASS: 보 휨 — phi_Mn={r1['phi_Mn_kNm']} > Mu=380, OK")
        passed += 1
    else:
        print(f"  FAIL: 보 휨 — {r1}")

    # 보 휨 NG 케이스
    r1b = check_beam_flexure(350, 560, 27, 400, 1000, 380)
    if r1b["check"] == "NG":
        print("  PASS: 철근 부족 시 NG 판정")
        passed += 1
    else:
        print("  FAIL: 철근 부족인데 OK 판정")

    # 보 전단 검토
    r2 = check_beam_shear(350, 560, 27, 400, 250, 10, 200)
    if "Vc_kN" in r2 and "Vs_kN" in r2 and "phi_Vn_kN" in r2:
        print(f"  PASS: 전단 — Vc={r2['Vc_kN']}, Vs={r2['Vs_kN']}, phi_Vn={r2['phi_Vn_kN']}")
        passed += 1
    else:
        print(f"  FAIL: 전단 결과 필드 누락")

    # 기둥 축력 검토
    r3 = check_column_axial(500, 500, 30, 400, 4053.6, 3000)
    if 0.01 <= r3["rho"] <= 0.08 and r3["rho_check"] == "OK":
        print(f"  PASS: 기둥 철근비 = {r3['rho']}, OK")
        passed += 1
    else:
        print(f"  FAIL: 기둥 철근비 — {r3}")

    # 기둥 축력 판정
    if r3["axial_check"] in ["OK", "NG"]:
        print(f"  PASS: 기둥 축력 판정 = {r3['axial_check']}")
        passed += 1
    else:
        print("  FAIL: 기둥 축력 판정 누락")

    # KDS 참조
    has_ref = all("reference" in r for r in [r1, r2, r3])
    if has_ref:
        print("  PASS: 모든 결과에 KDS 참조 포함")
        passed += 1
    else:
        print("  FAIL: KDS 참조 누락")

    print(f"\n결과: {passed}/6 통과")
    return passed == 6

verify_exercise2()

---
## 연습 3: 통합 건축공학 MCP 서버 (종합)

### 문제
연습 1 (BIM 데이터)과 연습 2 (구조 검토)를 통합하고, KDS 프롬프트를 추가하여 **완전한 건축공학 MCP 서버**를 `structural_mcp_server.py` 파일로 작성하세요.

**요구사항:**

| Primitive | 항목 | 설명 |
| --- | --- | --- |
| Resource | `bim://overview` | 건물 개요 |
| Resource | `bim://beams/{floor}` | 층별 보 목록 |
| Resource | `bim://columns/{floor}` | 층별 기둥 목록 |
| Resource | `kds://{code}/info` | KDS 기준 정보 |
| Tool | `check_beam_flexure(...)` | 보 휨 검토 |
| Tool | `check_beam_shear(...)` | 보 전단 검토 |
| Tool | `check_column_axial(...)` | 기둥 축력 검토 |
| Tool | `calc_concrete_volume(...)` | 물량 산출 |
| Prompt | `full_review(member_type)` | 종합 검토 프롬프트 |

**파일 저장 후 Inspector로 테스트:**
```bash
mcp dev structural_mcp_server.py
```

In [ ]:
# TODO: structural_mcp_server.py 파일을 작성하세요
# 연습 1 + 연습 2의 코드를 통합하고, 프롬프트를 추가합니다.

server_code = '''
import math
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("structural-engineering-server")

# TODO: BUILDING 데이터 + KDS 데이터
# TODO: Resources (bim://..., kds://...)
# TODO: Tools (check_beam_flexure, check_beam_shear, check_column_axial, calc_concrete_volume)
# TODO: Prompt (full_review)

if __name__ == "__main__":
    mcp.run()
'''

with open("structural_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("structural_mcp_server.py 저장 완료")
print("테스트: 'mcp dev structural_mcp_server.py'")

In [ ]:
# === 솔루션 ===

server_code = '''import math
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("structural-engineering-server")

# ========== 데이터 ==========

BUILDING = {
    "project": "경희대 연구동",
    "stories": [
        {"name": "1F", "level": 0.0, "height": 4.5},
        {"name": "2F", "level": 4.5, "height": 3.6},
        {"name": "3F", "level": 8.1, "height": 3.6},
    ],
    "beams": [
        {"id": "B1", "floor": "2F", "span": 8.0, "b": 350, "h": 600, "fck": 27},
        {"id": "B2", "floor": "2F", "span": 6.0, "b": 300, "h": 500, "fck": 27},
        {"id": "B3", "floor": "3F", "span": 8.0, "b": 350, "h": 600, "fck": 24},
    ],
    "columns": [
        {"id": "C1", "floor": "1F", "b": 500, "h": 500, "fck": 30, "Pu": 3000},
        {"id": "C2", "floor": "1F", "b": 400, "h": 400, "fck": 27, "Pu": 1800},
    ]
}

KDS_DATA = {
    "14-20-20": {
        "title": "콘크리트구조 일반설계기준",
        "sections": {
            "4.2.3": "설계휨강도: phi*Mn >= Mu, phi=0.85",
            "4.3.2": "처짐 검토: 보 최소 두께 L/16 (단순보)",
            "3.2.1": "기둥 철근비: 0.01 <= rho <= 0.08",
        }
    },
    "14-20-22": {
        "title": "전단 및 비틀림 설계기준",
        "sections": {
            "5.3.1": "Vc = (1/6)*sqrt(fck)*b*d",
            "5.4.2": "최소 전단철근: Av,min = 0.062*sqrt(fck)*b*s/fy",
        }
    }
}

# ========== Resources ==========

@mcp.resource("bim://overview")
def get_overview() -> str:
    """건물 개요를 반환합니다."""
    return (
        f"{BUILDING[\'project\']}: "
        f"{len(BUILDING[\'stories\'])}개 층, "
        f"보 {len(BUILDING[\'beams\'])}개, "
        f"기둥 {len(BUILDING[\'columns\'])}개"
    )

@mcp.resource("bim://beams/{floor}")
def get_beams(floor: str) -> str:
    """특정 층의 보 목록을 반환합니다."""
    beams = [b for b in BUILDING["beams"] if b["floor"] == floor.upper()]
    if not beams:
        return f"{floor}층에 보 없음"
    lines = [f"{floor}층 보 ({len(beams)}개):"]
    for b in beams:
        lines.append(f"  {b[\'id\']}: {b[\'span\']}m, {b[\'b\']}x{b[\'h\']}mm, fck={b[\'fck\']}")
    return "\\n".join(lines)

@mcp.resource("bim://columns/{floor}")
def get_columns(floor: str) -> str:
    """특정 층의 기둥 목록을 반환합니다."""
    cols = [c for c in BUILDING["columns"] if c["floor"] == floor.upper()]
    if not cols:
        return f"{floor}층에 기둥 없음"
    lines = [f"{floor}층 기둥 ({len(cols)}개):"]
    for c in cols:
        lines.append(f"  {c[\'id\']}: {c[\'b\']}x{c[\'h\']}mm, fck={c[\'fck\']}, Pu={c[\'Pu\']}kN")
    return "\\n".join(lines)

@mcp.resource("kds://{code}/info")
def get_kds(code: str) -> str:
    """KDS 기준 정보를 반환합니다."""
    if code not in KDS_DATA:
        return f"알 수 없는 코드: {code}"
    kds = KDS_DATA[code]
    sections = "\\n".join(f"  {k}: {v}" for k, v in kds["sections"].items())
    return f"KDS {code} {kds[\'title\']}\\n{sections}"

# ========== Tools ==========

@mcp.tool()
def check_beam_flexure(b: float, d: float, fck: float, fy: float, As: float, Mu: float) -> dict:
    """RC 보의 휨 강도를 검토합니다 (KDS 14 20 20)."""
    a = (As * fy) / (0.85 * fck * b)
    Mn = As * fy * (d - a / 2) / 1e6
    phi_Mn = 0.85 * Mn
    return {
        "a_mm": round(a, 1), "Mn_kNm": round(Mn, 1),
        "phi_Mn_kNm": round(phi_Mn, 1), "Mu_kNm": Mu,
        "check": "OK" if phi_Mn >= Mu else "NG",
        "reference": "KDS 14 20 20, 4.2.3"
    }

@mcp.tool()
def check_beam_shear(b: float, d: float, fck: float, fy: float, Vu: float, stirrup_d: float, stirrup_s: float) -> dict:
    """RC 보의 전단 강도를 검토합니다 (KDS 14 20 22)."""
    Vc = (1/6) * math.sqrt(fck) * b * d / 1000
    Av = 2 * math.pi * (stirrup_d / 2) ** 2
    Vs = Av * fy * d / (stirrup_s * 1000)
    phi_Vn = 0.75 * (Vc + Vs)
    return {
        "Vc_kN": round(Vc, 1), "Vs_kN": round(Vs, 1),
        "phi_Vn_kN": round(phi_Vn, 1), "Vu_kN": Vu,
        "check": "OK" if phi_Vn >= Vu else "NG",
        "reference": "KDS 14 20 22"
    }

@mcp.tool()
def check_column_axial(b: float, h: float, fck: float, fy: float, Ast: float, Pu: float) -> dict:
    """RC 기둥의 축력을 검토합니다 (KDS 14 20 20)."""
    Ag = b * h
    rho = Ast / Ag
    phi_Pn = 0.80 * 0.65 * (0.85 * fck * (Ag - Ast) + fy * Ast) / 1000
    return {
        "rho": round(rho, 4),
        "rho_check": "OK" if 0.01 <= rho <= 0.08 else "NG",
        "phi_Pn_kN": round(phi_Pn, 1), "Pu_kN": Pu,
        "axial_check": "OK" if Pu <= phi_Pn else "NG",
        "reference": "KDS 14 20 20"
    }

@mcp.tool()
def calc_concrete_volume(member_type: str, floor: str = "all") -> dict:
    """콘크리트 물량을 산출합니다."""
    if member_type == "beam":
        members = BUILDING["beams"]
        if floor != "all":
            members = [m for m in members if m["floor"] == floor.upper()]
        total = sum(m["b"]/1000 * m["h"]/1000 * m["span"] for m in members)
    elif member_type == "column":
        members = BUILDING["columns"]
        if floor != "all":
            members = [m for m in members if m["floor"] == floor.upper()]
        heights = {s["name"]: s["height"] for s in BUILDING["stories"]}
        total = sum(m["b"]/1000 * m["h"]/1000 * heights.get(m["floor"], 3.6) for m in members)
    else:
        return {"error": "beam or column"}
    return {"member_type": member_type, "floor": floor, "count": len(members), "volume_m3": round(total, 3)}

# ========== Prompt ==========

@mcp.prompt()
def full_review(member_type: str) -> str:
    """종합 구조 검토 프롬프트"""
    if member_type == "beam":
        return (
            "RC 보의 종합 설계 검토를 수행하세요.\\n\\n"
            "1. bim://beams/{floor}에서 보 정보를 확인\\n"
            "2. check_beam_flexure로 휨 강도 검토\\n"
            "3. check_beam_shear로 전단 강도 검토\\n"
            "4. 결과를 요약표로 정리\\n"
            "5. 부적합 시 개선안 제시"
        )
    elif member_type == "column":
        return (
            "RC 기둥의 종합 설계 검토를 수행하세요.\\n\\n"
            "1. bim://columns/{floor}에서 기둥 정보를 확인\\n"
            "2. check_column_axial로 축력 검토\\n"
            "3. 결과를 요약표로 정리"
        )
    return "지원: beam, column"

if __name__ == "__main__":
    mcp.run()
'''

with open("structural_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("structural_mcp_server.py 저장 완료!")
print("\n테스트: 'mcp dev structural_mcp_server.py'")
print("\nInspector에서 확인할 항목:")
print("  Resources: bim://overview, bim://beams/2F, kds://14-20-20/info")
print("  Tools: check_beam_flexure, check_beam_shear, check_column_axial")
print("  Prompts: full_review(beam), full_review(column)")

In [ ]:
# 검증 함수
import os

def verify_exercise3():
    if not os.path.exists("structural_mcp_server.py"):
        print("  FAIL: structural_mcp_server.py 파일 없음")
        return False

    with open("structural_mcp_server.py", "r", encoding="utf-8") as f:
        content = f.read()

    checks = [
        ("FastMCP" in content, "FastMCP 사용"),
        ("@mcp.resource" in content, "리소스 정의"),
        ("@mcp.tool" in content, "도구 정의"),
        ("@mcp.prompt" in content, "프롬프트 정의"),
        ("bim://overview" in content or "bim://" in content, "BIM 리소스"),
        ("kds://" in content, "KDS 리소스"),
        ("check_beam_flexure" in content, "보 휨 도구"),
        ("check_beam_shear" in content, "보 전단 도구"),
        ("check_column_axial" in content, "기둥 도구"),
        ("calc_concrete_volume" in content, "물량 도구"),
        ("full_review" in content, "검토 프롬프트"),
        ("mcp.run()" in content, "서버 실행"),
        ("BUILDING" in content, "BIM 데이터"),
        ("KDS" in content, "KDS 데이터"),
    ]

    passed = 0
    for check, name in checks:
        if check:
            print(f"  PASS: {name}")
            passed += 1
        else:
            print(f"  FAIL: {name}")

    print(f"\n결과: {passed}/{len(checks)} 통과")

    if passed == len(checks):
        print("\n축하합니다! 완전한 건축공학 MCP 서버를 작성했습니다.")
        print("다음 단계: 터미널에서 'mcp dev structural_mcp_server.py' 실행")
        print("Inspector에서 모든 Tools, Resources, Prompts를 테스트하세요.")

    return passed == len(checks)

verify_exercise3()